In [1]:
# =====================================================================
# ANIMATION SETTINGS
# =====================================================================

fps = 10


# =====================================================================

import pyvista as pv
import numpy as np
import meshio
import shutil
from pathlib import Path


def read_result(filename, timestep=None):
    """
    Read time-dependent result XDMF with PyVista.
    """

    filename = str(filename)
    reader = pv.get_reader(filename)

    if hasattr(reader, "time_values"):
        time_values = reader.time_values

        if len(time_values) > 0:
            if timestep is None:
                timestep = -1

            if timestep < 0:
                timestep = len(time_values) + timestep

            if timestep < 0 or timestep >= len(time_values):
                raise ValueError(
                    f"Invalid timestep {timestep}. "
                    f"Available steps: 0 to {len(time_values) - 1}"
                )

            reader.set_active_time_point(timestep)

    mesh = reader.read()

    if isinstance(mesh, pv.MultiBlock):
        blocks = []

        for block in mesh:
            if block is not None and block.n_cells > 0:
                blocks.append(block)

        if not blocks:
            raise RuntimeError(
                f"No valid mesh blocks found in {filename}"
            )

        if len(blocks) == 1:
            mesh = blocks[0]
        else:
            mesh = pv.MultiBlock(blocks).combine()

    return mesh


def count_timesteps(filename):
    """
    Number of time points stored in the result XDMF.
    """

    reader = pv.get_reader(str(filename))

    if hasattr(reader, "time_values"):
        n_steps = len(reader.time_values)

        if n_steps > 0:
            return n_steps

    return 1


def read_outline(filename):
    """
    Read static outline XDMF using meshio.

    This avoids the PyVista XDMF reader issue for XDMF files
    that contain no time information.
    """

    mesh = meshio.read(filename)

    points = np.asarray(mesh.points)

    line_cells = None

    for cell_block in mesh.cells:
        if cell_block.type == "line":
            line_cells = np.asarray(cell_block.data)
            break

    if line_cells is None:
        raise RuntimeError(
            "No line elements found in outline XDMF."
        )

    # PyVista line connectivity format:
    #
    # [2, p0, p1,
    #  2, p0, p1,
    #  ...]
    #
    lines = np.hstack(
        [
            np.full(
                (line_cells.shape[0], 1),
                2,
                dtype=np.int64,
            ),
            line_cells.astype(np.int64),
        ]
    ).ravel()

    outline = pv.PolyData(
        points,
        lines=lines,
    )

    return outline


def get_scalar_range(mesh, scalar_name):
    if scalar_name in mesh.point_data:
        values = mesh.point_data[scalar_name]

    elif scalar_name in mesh.cell_data:
        values = mesh.cell_data[scalar_name]

    else:
        available = (
            list(mesh.point_data.keys())
            + list(mesh.cell_data.keys())
        )

        raise KeyError(
            f"Scalar '{scalar_name}' not found.\n"
            f"Available arrays: {available}"
        )

    return np.nanmin(values), np.nanmax(values)


def visualize_damage(
    result_file,
    outline_file,
    timestep=-1,
    damage_name="damage",
    clip_value=0.0,
    invert=True,
    screenshot="damage_view.png",
    window_size=(1600, 1000),
    cmap="viridis",
    full_domain_opacity=0.3,
    outline_width=2.0,
    zoom=1.0,
    color_range=None,
    outline=None,
):
    # -----------------------------------------------------------------
    # Read result
    # -----------------------------------------------------------------

    result = read_result(
        result_file,
        timestep=timestep,
    )

    # -----------------------------------------------------------------
    # Read separate outline
    #
    # The outline is static, so an already-loaded one can be passed in
    # and reused across frames.
    # -----------------------------------------------------------------

    if outline is None:
        outline = read_outline(
            outline_file,
        )

    # -----------------------------------------------------------------
    # Damage range
    #
    # `color_range` holds it fixed across the animation; without it the
    # range is taken per frame, exactly as in render.ipynb.
    # -----------------------------------------------------------------

    if color_range is None:
        damage_min, damage_max = get_scalar_range(
            result,
            damage_name,
        )
    else:
        damage_min, damage_max = color_range

    # -----------------------------------------------------------------
    # Scalar clip
    # -----------------------------------------------------------------

    damage_clip = result.clip_scalar(
        scalars=damage_name,
        value=clip_value,
        invert=invert,
    )

    # -----------------------------------------------------------------
    # Plotter
    # -----------------------------------------------------------------

    plotter = pv.Plotter(
        off_screen=True,
        window_size=window_size,
    )

    plotter.set_background("white")

    # -----------------------------------------------------------------
    # Full domain
    # -----------------------------------------------------------------

    plotter.add_mesh(
        result,
        # color="white",
        scalars=damage_name,
        cmap=cmap,
        opacity=full_domain_opacity,
        show_scalar_bar=False,
        show_edges=False,
        lighting=True,
    )

    # -----------------------------------------------------------------
    # Damage clip
    # -----------------------------------------------------------------

    # Early frames can have no damage above the clip value, which leaves an
    # empty mesh carrying no arrays; adding it would raise a KeyError.
    if damage_clip.n_cells > 0:
        plotter.add_mesh(
            damage_clip,
            scalars=damage_name,
            cmap=cmap,
            clim=(damage_min, damage_max),
            show_scalar_bar=False,
            lighting=True,
        )

    # -----------------------------------------------------------------
    # Outline
    # -----------------------------------------------------------------

    plotter.add_mesh(
        outline,
        color="black",
        line_width=outline_width,
        lighting=False,
    )

    # -----------------------------------------------------------------
    # Camera
    # -----------------------------------------------------------------

    bounds = result.bounds

    xmin, xmax = bounds[0], bounds[1]
    ymin, ymax = bounds[2], bounds[3]
    zmin, zmax = bounds[4], bounds[5]

    center = np.array(
        [
            0.5 * (xmin + xmax),
            0.5 * (ymin + ymax),
            0.5 * (zmin + zmax),
        ]
    )

    dx = xmax - xmin
    dy = ymax - ymin
    dz = zmax - zmin

    diagonal = np.sqrt(
        dx**2
        + dy**2
        + dz**2
    )

    direction = np.array(
        [
            1.0,
            -1.0,
            1.0,
        ]
    )

    direction /= np.linalg.norm(
        direction
    )

    camera_distance = 2.0 * diagonal

    camera_position = (
        center
        + camera_distance * direction
    )

    plotter.camera_position = [
        camera_position,
        center,
        (0.0, 0.0, 1.0),
    ]

    plotter.camera.up = (
        0.0,
        0.0,
        1.0,
    )

    plotter.reset_camera()
    # Rotate view +90 degrees about Z
    plotter.camera.Azimuth(-90)
    if zoom != 1.0:
        plotter.camera.zoom(zoom)

    # -----------------------------------------------------------------
    # Screenshot
    # -----------------------------------------------------------------

    screenshot = Path(screenshot)

    screenshot.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    plotter.screenshot(
        str(screenshot)
    )

    plotter.close()

    return screenshot


def write_movie(frame_files, movie_path, fps):
    """
    Combine rendered frames into a movie at the given frame rate.
    """

    try:
        import imageio.v2 as imageio

    except ImportError as exc:
        raise RuntimeError(
            "Writing the movie needs imageio:\n"
            "    pip install imageio imageio-ffmpeg"
        ) from exc

    movie_path = Path(movie_path)

    movie_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if movie_path.suffix.lower() == ".gif":
        writer = imageio.get_writer(
            str(movie_path),
            fps=fps,
            loop=0,
        )

    else:
        # macro_block_size=1 keeps the exact window size instead of
        # padding up to the next multiple of 16.
        writer = imageio.get_writer(
            str(movie_path),
            fps=fps,
            macro_block_size=1,
        )

    for frame_file in frame_files:
        writer.append_data(
            imageio.imread(str(frame_file))
        )

    writer.close()

    return movie_path


# =====================================================================
# USER SETTINGS
# =====================================================================

try:
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib


def find_outline(subfolder_name):
    """
    Scan TOML files in the parent directory, find the one whose output
    directory matches subfolder_name, and return the outline XDMF path
    derived from the mesh path in that TOML.
    """
    toml_dir = Path("..")
    for toml_file in sorted(toml_dir.glob("*.toml")):
        with open(toml_file, "rb") as f:
            config = tomllib.load(f)
        out_dir = config.get("output", {}).get("directory", "")
        if Path(out_dir.rstrip("/")).name == subfolder_name:
            mesh_path = config["mesh"]["path"]
            full_mesh = (toml_dir / mesh_path).resolve()
            outline = full_mesh.with_name(
                full_mesh.stem + "_outline" + full_mesh.suffix
            )
            return outline
    return None


damage_variable = "damage"
damage_clip_value = 0.99

movie_suffix = ".mp4"       # ".gif" also works
lock_color_range = True     # False reproduces render.ipynb per-frame range
keep_frames = False         # True keeps the PNGs after the movie is written


# =====================================================================
# RUN
# =====================================================================

here = Path(".")
result_files = sorted(here.glob("*/output.xdmf"))

for result_file in result_files:
    subfolder = result_file.parent
    outline_file = find_outline(subfolder.name)

    if outline_file is None:
        print(f"WARNING: no TOML found for subfolder '{subfolder.name}', skipping.")
        continue

    if not outline_file.exists():
        print(f"WARNING: outline not found at {outline_file}, skipping.")
        continue

    n_steps = count_timesteps(result_file)

    print(f"\n{'='*60}")
    print(f"Processing: {result_file}")
    print(f"Outline:    {outline_file}")
    print(f"Timesteps:  {n_steps}")
    print(f"{'='*60}")

    # The outline never changes, so read it once for the whole sequence.
    outline_mesh = read_outline(outline_file)

    # Take the colour range from the final step, where damage is largest,
    # so the colours do not shift from frame to frame.
    color_range = None

    if lock_color_range:
        color_range = get_scalar_range(
            read_result(result_file, timestep=-1),
            damage_variable,
        )
        print(f"Colour range locked to {color_range}")

    frames_dir = subfolder / "frames"
    frame_files = []

    for step in range(n_steps):
        frame_file = frames_dir / f"damage_step_{step:04d}.png"

        visualize_damage(
            result_file=str(result_file),
            outline_file=str(outline_file),
            timestep=step,
            damage_name=damage_variable,
            clip_value=damage_clip_value,
            invert=False,
            screenshot=str(frame_file),
            window_size=(1600, 1000),
            cmap="jet",
            full_domain_opacity=0.05,
            outline_width=2.0,
            zoom=1.0,
            color_range=color_range,
            outline=outline_mesh,
        )

        frame_files.append(frame_file)

        print(f"  frame {step + 1:4d} / {n_steps}", end="\r")

    print()

    movie_file = subfolder / f"damage{movie_suffix}"

    write_movie(frame_files, movie_file, fps)

    duration = len(frame_files) / float(fps)

    print(
        f"Movie written to: {movie_file} "
        f"({len(frame_files)} frames at {fps} fps, {duration:.1f} s)"
    )

    # Delete the frames folder now that the movie exists. rmtree rather than
    # rmdir, so a stray file such as .DS_Store cannot block the cleanup.
    if not keep_frames:
        shutil.rmtree(frames_dir, ignore_errors=True)

        print(f"Frames folder deleted: {frames_dir}")



Processing: 01/output.xdmf
Outline:    /home/abhi/adaptive-phase-field-fracture/data/mesh/07/Lx500_5C_S50_outline.xdmf
Timesteps:  93
Colour range locked to (-7.397989e-18, 1.0)


2026-09-12 23:10:55.784 (   0.784s) [    75FA6B018740]vtkXOpenGLRenderWindow.:1460  WARN| bad X server connection. DISPLAY=


  frame   93 / 93
Movie written to: 01/damage.mp4 (93 frames at 10 fps, 9.3 s)
Frames folder deleted: 01/frames

Processing: 02/output.xdmf
Outline:    /home/abhi/adaptive-phase-field-fracture/data/mesh/08/Lx500_5C_S70_outline.xdmf
Timesteps:  93
Colour range locked to (-4.980781e-18, 1.0)
  frame   93 / 93
Movie written to: 02/damage.mp4 (93 frames at 10 fps, 9.3 s)
Frames folder deleted: 02/frames

Processing: 03/output.xdmf
Outline:    /home/abhi/adaptive-phase-field-fracture/data/mesh/09/Lx500_10C_S70_outline.xdmf
Timesteps:  63
Colour range locked to (-7.3737217e-19, 1.0)
  frame   63 / 63
Movie written to: 03/damage.mp4 (63 frames at 10 fps, 6.3 s)
Frames folder deleted: 03/frames
